## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 3b: Învățare nesupervizată: asociere și recomandări</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/03-invatare-nesupervizata/curs-03b-asociere-recomandari.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/03-invatare-nesupervizata
sys.path.insert(0, os.path.abspath("../.."))

<b>Obiectivele cursului:</b>

Să înțelegeți conceptele de bază ale învățării nesupervizate.

Să explorați și să aplicați algoritmi de generare de recomandări.

Să analizați și să interpretați rezultatele învățării nesupervizate.

<b>Structura cursului:</b>
- Analiza asocierilor și recomandări
    - algoritmul Apriori
- Sisteme de recomandare
    - LightFM

# Analiza asocierilor și recomandări


## APRIORI și Seturile de articole frecvente (<a href ="https://www.geeksforgeeks.org/frequent-item-set-in-data-set-association-rule-mining/">Asociation Rule Mining</a>)

Seturile de articole frecvente (<b>frecvent item set</b>), cunoscute și sub numele de reguli de asociere, sunt un concept fundamental în extragerea regulilor de asociere. Aceasta este o tehnică utilizată pentru a descoperi relațiile dintre elementele care apar frecvent împreună.

`Discuție` - încercați să indicați despre care din aplicațiile de mai jos stiți că folosesc recomandări:
<img style="float: center; padding-right: 10px; width: 600px" src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/recomandari1.jpg" >
<center><b>Fig. Exemple de aplicații care folosesc recomandări</b> </center>

Un set de articole frecvente este un set de elemente care apar împreună frecvent într-un set de date. Frecvența unui set de articole este măsurată prin <b>suport</b>, care este numărul de tranzacții sau înregistrări din setul de date care conține setul de articole. De exemplu, dacă un set de date conține 100 de tranzacții și setul de articole <i>{lapte, pâine}</i> apare în 20 dintre aceste tranzacții, suportul pentru {lapte, pâine} este 20.

Algoritmii de extragere a regulilor de asociere, cum ar fi Apriori sau FP-Growth, sunt utilizați pentru a găsi seturi  de articole frecvente și pentru a genera reguli de asociere. Acești algoritmi funcționează prin generarea iterativă de seturi de articole candidate și eliminarea celor care nu îndeplinesc pragul minim de suport. Odată găsite seturile de articole frecvente, regulile de asociere pot fi generate utilizând conceptul de încredere, care este raportul dintre numărul de tranzacții care conțin setul de articole și numărul de tranzacții care conțin antecedentul (partea stângă) a regulii.

Seturile frecvente de articole și regulile de asociere pot fi utilizate pentru o varietate de sarcini, cum ar fi analiza coșului de cumpărături, vânzări încrucișate și sisteme de recomandare. Cu toate acestea, trebuie remarcat faptul că extragerea regulilor de asociere poate genera un număr mare de reguli, dintre care multe pot fi irelevante sau neinteresante. Prin urmare, este important să folosiți măsuri adecvate, cum ar fi <b>lift</b> și <b>încredere (conviction)</b>, pentru a evalua importanța regulilor generate. Le vom vedeam mai jos.

Dacă există 2 articole X și Y achiziționate frecvent, atunci este bine să le puneți împreună în magazine sau să oferiți o reducere la un articol la achiziționarea altui articol. Acest lucru poate crește cu adevărat vânzările. De exemplu, este probabil să se constate că, dacă un client cumpără lapte și pâine, atunci cumpără și unt. Deci regula de asociere este [‘lapte]^[‘pâine’]→[‘unt’]. Astfel, vânzătorul poate sugera clientului să cumpere unt dacă el/ea cumpără lapte și pâine. Mai subtil, dacă magazinul constată că majoritatea cumpărătorilor lapte și pâine cumpără și unt, atunci poate aranja magazinul încât raioanele să fie:
- apropiate, dacă scopul este o experiență plăcută a cumpărătorului frecvent (de exemplu raioanele din Auchan, Carrefour, dar și Dedeman, etc.)
- depărtate unele de altele, pentru a lăsa cumpărăturile impulsive să acționeze pe drumul dintre cele două raioane.

<b>Câteva definiții</b>
- Support_count = Suportul unui element: numărul de tranzacții în care apare.
- Suport unei reguli: Este una dintre măsurile de interes. Aceasta spune despre utilitatea și certitudinea regulilor. Suport de 5% înseamnă că 5% din tranzacțiile din baza de date respectă regula.
    - Suport(A → B) = Suport_count(A ∪ B)
- Încredere: O încredere de 60% înseamnă că 60% dintre clienții care au cumpărat lapte și pâine au cumpărat și unt.
    - Încredere(A → B) = Suport_count(A ∪ B) / Support_count(A)
- Lift: Lift(A → B) se referă la creșterea raportului de vânzare a lui B atunci când este vândut A. Lift(A → B) poate fi calculată împărțind Încrederea (A -> B) la Suport(B):
    $$Lift(A→B) = Încrederea (A→B)/Suport (B)$$
- Regulă puternică: Dacă o regulă are atât suportul >= minim, cât și încrederea >= minim, atunci este o regulă puternică.
- Support_count(X): Numărul de tranzacții în care apare X. Dacă X este $A\cup B$, atunci este numărul de tranzacții în care A și B sunt ambele prezente.
- Setul maxim de articole: un set de articole frecvente este maxim dacă niciunul dintre superseturile sale nu este frecvent.
- Set de articole închis: Un set de articole este închis într-un set de date dacă nu există niciun superset frecvent care să aibă același suport ca și acest set de articole original.
- K- Itemset: Setul de articole care conține K articole este un K-itemset. Deci se poate spune că un set de articole este frecvent dacă numărul de suport corespunzător este mai mare decât numărul minim de suport.

Să presupunem că avem 1000 de tranzacții și dorim să găsim suportul, încrederea și lift-ul pentru două articole, de ex. Burgeri și Ketchup. Din o mie de tranzacții, 100 conțin ketchup, în timp ce 150 conțin un burger. Din 150 de tranzacții în care este achiziționat un burger, 50 de tranzacții conțin și ketchup. Folosind aceste date, dorim să găsim sprijinul, încrederea și creșterea.

Suportul se referă la popularitatea implicită a unui articol și poate fi calculat prin găsirea numărului de tranzacții care conțin un anumit articol împărțit la numărul total de tranzacții. Să presupunem că vrem să găsim suport pentru elementul B. Acesta poate fi calculat ca:

Support(Ketchup) = (Transacții conținând Ketchup)/(Total Transacții)

Support(Ketchup) = 100/1000
                 = 10%

Încrederea se referă la probabilitatea ca și un articol B să fie cumpărat dacă este cumpărat articolul A. Poate fi calculat prin găsirea numărului de tranzacții în care A și B sunt cumpărate împreună, împărțit la numărul total de tranzacții în care A este cumpărat. Revenind la problema noastră, am avut 50 de tranzacții în care Burger și Ketchup au fost cumpărate împreună. În timp ce în 150 de tranzacții se cumpără burgeri. Apoi putem găsi că probabilitatea de a cumpăra ketchup atunci când un burger este cumpărat poate fi reprezentată ca încredere a Burger → Ketchup și poate fi scrisă matematic ca:

Confidence(Burger→Ketchup) = (Transacții conținând  (Burger și Ketchup))/(Transacții conținând Burger)

Confidence(Burger→Ketchup) = 50/150
                           = 33.3%

Lift(A -> B) se referă la creșterea raportului de vânzare a lui B atunci când este vândut A. Lift(A –> B) poate fi calculată împărțind Încrederea (A -> B) împărțită la Suport(B). Revenind la problema noastră cu Burger și Ketchup, Lift (Burger -> Ketchup) poate fi calculată astfel:

Lift(Burger→Ketchup) = (Confidence (Burger→Ketchup))/(Suport (Ketchup))

Lift(Burger→Ketchup) = 33,3/10
                      = 3,33
                      
Lift, practic, ne spune că probabilitatea de a cumpăra un Burger și Ketchup împreună este de 3,33 ori mai mare decât probabilitatea de a cumpăra doar ketchup. 
- Lift = 1 înseamnă că nu există o asociere între produsele A și B. 
- Lift > 1 înseamnă că este probabil ca produsele A și B  să fie cumpărate împreună.
- Lift < 1 se referă la cazul în care este puțin probabil ca cele două produse  să fie cumpărate împreună.

#### Exemplu pentru găsirea seturilor de articole frecvente

Considerăm tranzacțiile (pot fi filme vizionate de un client de streaming, melodii ascultate, produse cumpărate, pagini vizitate, cărți citite, reel-uri la care ați poposit mai mult de 5 secunde, etc.):

<img style="float: center; padding-right: 10px; width: 300px" src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/frq_itemset_example.png" >
<center><b>Fig. Exemplu mulțimi frecvente</b> (Credits <a href ="https://www.geeksforgeeks.org/apriori-algorithm/" >geeksforgeeks</a>)</center>

Să presupunem că <b>suportul minim = 3</b> pentru tranzacțiile din figură. <b>Support_count</b> numără aparițiile unei mulțimi considerate ca și submulțime a mulțimilor din tranzacții.

<b>1-itemset frecvent:</b>
- Support_count({A}) = 3 - A apare în 3 tranzacții; // nu e închis din cauza lui {A, C} și nici maximal
- Support_count({B}) = 4; // nu e închis din cauza lui  {B, D} și nici maximal 
- Support_count({C}) = 4; // nu e închis din cauza lui  {C, D} și nici maximal 
- Support_count({D}) = 5; // închis pentru că nu există supramulțime cu același suport. Nu e maximal pentru că există superseturi frecvente.

<b>2-itemset frecvent</b>: 
- Support_count({A, B}) = 2 // setul nu e frecvent deoarece support_count = 2 < 3 = minimul de support_count
- Support_count({A, C}) = 3 // nu e închis din cauza lui  {A, C, D} 
- Support_count({A, D}) = 3 // nu e închis din cauza lui {A, C, D} 
- Support_count({B, C}) = 3 // nu e închis din cauza lui  {B, C, D} 
- Support_count({B, D}) = 4 // închis dar nu maximal in cauza lui {B, C, D} 
- Support_count({C, D}) = 4 // închis dar nu maximal in cauza lui {B, C, D} 

<b>3-itemset frecvent:</b> 
- Support_count({A, B, C}) = 2 // setul nu e frecvent deoarece support_count < 3 = minimul de support_count
- Support_count({A, B, D}) = 2 // setul nu e frecvent deoarece support_count < 3 = minimul de support_count
- Support_count({A, C, D}) = 3 // maximal frecvent 
- Support_count({B, C, D}) = 3 // maximal frecvent 

<b>4-itemset frecvent</b>: 
- Support_count({A, B, C, D}) = 2 //nu este frecvent

### Pași implicați în algoritmul apriori
Pentru seturi mari de date, pot exista sute de articole în sute de mii de tranzacții. Algoritmul Apriori încearcă să extragă reguli pentru fiecare combinație posibilă de elemente. De exemplu, Creșterea poate fi calculată pentru articolul 1 și articolul 2, articolul 1 și articolul 3, articolul 1 și articolul 4 și apoi articolul 2 și articolul 3, articolul 2 și articolul 4 și apoi combinațiile de articole, de ex. articolele 1, 2 și 3; în mod similar articolul 1, articolul 2 și articolul 4 și așa mai departe.

După cum puteți vedea din exemplul de mai sus, acest proces poate fi extrem de lent din cauza numărului de combinații. Pentru a accelera procesul, trebuie să parcurgem următorii pași:

1. Setați o valoare minimă pentru sprijin și încredere. Aceasta înseamnă că suntem interesați doar să găsim reguli pentru articolele care au o anumită valoare măsurabilă pentru existență implicită (de exemplu, suport) și au o valoare minimă pentru apariția concomitentă cu alte elemente (de exemplu, încredere).
2. Extrageți toate subseturile care au o valoare mai mare a suportului decât pragul minim.
3. Selectați toate regulile din subseturile cu valoare de încredere mai mare decât pragul minim.
4. Ordonați regulile în ordinea descrescătoare a lift-ului.

### AVANTAJE ȘI DEZAVANTAJE

### Avantajele utilizării frecvente a seturilor de articole și a extragerii regulilor de asociere includ:
- <b>Descoperirea eficientă a modelelor:</b> algoritmii de extragere a regulilor de asociere sunt eficienți în descoperirea modelelor în seturi mari de date, făcându-le utili pentru sarcini precum analiza coșului de piață și sistemele de recomandare.
- <b>Ușor de interpretat:</b> rezultatele extragerii regulilor de asociere sunt ușor de înțeles și interpretat, făcând posibilă explicarea tiparelor găsite în date.
- <b>Poate fi utilizat într-o gamă largă de aplicații:</b> Exploatarea regulilor de asociere poate fi utilizată într-o gamă largă de aplicații, cum ar fi retail, finanțe și asistență medicală, ceea ce poate ajuta la îmbunătățirea procesului decizional și la creșterea veniturilor.
- <b>Gestionarea seturilor mari de date:</b> acești algoritmi pot gestiona seturi mari de date cu multe articole și tranzacții, ceea ce le face potrivite pentru scenariile de date mari.

### Dezavantajele utilizării frecvente a seturilor de articole și a extragerii regulilor de asociere includ:
- <b>Număr mare de reguli generate:</b> Exploatarea regulilor de asociere poate genera un număr mare de reguli, dintre care multe pot fi irelevante sau neinteresante, ceea ce poate face dificilă identificarea celor mai importante modele.
- <b>Limitat în detectarea relațiilor complexe: </b>Exploatarea regulilor de asociere este limitată în capacitatea sa de a detecta relații complexe între articole și ia în considerare doar apariția simultană a articolelor în aceeași tranzacție.
- <b>Poate fi costisitor din punct de vedere computațional:</b> pe măsură ce crește numărul de articole și tranzacții, crește și numărul de seturi de articole candidate, ceea ce poate face algoritmul costisitor din punct de vedere computațional.
- <b>Necesitatea de a defini pragul minim de suport și încredere:</b> pragul minim de suport și încredere trebuie stabilit înainte de procesul de extragere a regulilor de asociere, care poate fi dificil și necesită o bună înțelegere a datelor

`Gând la final` Imaginati-vă cantitatea de date legate de urmele lăsate de fiecare din nou prin aplicațiile listate la început, precum și numărul total de itemuri, și seturi de itemuri care ar trebui considerate în cazul acestor mega-platforme. 

### Algoritmul APRIORI

Algoritmul Apriori a fost prezentat de R. Agrawal și R. Srikant în 1994 pentru a găsi seturi de articole frecvente într-un set de date folosind regula de asociere booleană. Numele algoritmului este Apriori deoarece folosește informațiile anterioare despre proprietățile frecvente ale seturilor de articole. Aplicăm o abordare iterativă sau o căutare pe nivel în care se folosesc k-seturi de articole frecvente pentru a găsi k+1 seturi de articole.

Pentru a îmbunătăți eficiența generării la nivel de seturi de articole frecvente, este utilizată o proprietate importantă numită <b>proprietatea Apriori</b>, care ajută prin reducerea spațiului de căutare.

<b>Proprietatea Apriori</b>
<i>Toate submulțimile nevide ale setului de articole frecvente sunt frecvente</i>. 
Conceptul cheie al algoritmului Apriori este această măsura antimonotonică a suportului. Reciproc,
<i>dacă un set de articole nu este frecvent, nici-unul din superseturile sale nu este frecvent.</i>

Teoretic algoritmul Apriori generează toate itemseturile frecvente. Pentru o mulțime mare poate dura extrem de mult și ne putem limita la K-itemseturi frecvent, pentru un K care să fie convenabil (2,3 de exemplu). Pentru detalii asupra pașilor algoritmului puteși consulta de exemplu <a href ="https://www.geeksforgeeks.org/apriori-algorithm/">sursa asta</a>.

<b>Confidence – Încrederea</b>
O încredere de 60% înseamnă că 60% dintre clienții care au cumpărat lapte și pâine au cumpărat și unt.


### Limitările algoritmului apriori
Algoritmul apriori poate fi lent. Principala limitare este timpul necesar pentru a deține un număr mare de seturi candidate cu seturi de articole foarte frecvente, suport minim scăzut sau seturi mari de articole; nu este o abordare eficientă pentru un număr mare de seturi de date. De exemplu, dacă există 10^4 seturi frecvente de 1 articol, trebuie să genereze mai mult de 10^7 candidați d mărime 2, care la rândul lor vor fi testați și acumulați. În plus, pentru a detecta modele frecvente de lungime 100, adică v1, v2... v100, trebuie să genereze 2^100 seturi candidate, care produc costuri și pierderi de timp pentru generarea de candidați. Deci, va verifica multe seturi din seturile de articole candidate, de asemenea, va scana baza de date de multe ori în mod repetat pentru a găsi seturi de articole candidate. Apriori va fi foarte scăzut și ineficienț atunci când capacitatea de memorie este limitată și avem de a face cu un număr mare de tranzacții. [Sursa: https://arxiv.org/pdf/1403.3948.pdf]

## RECSYS - Generatoare de recomandări (<a href ="https://towardsdatascience.com/recommender-systems-a-complete-guide-to-machine-learning-models-96d3f94ea748">ref.</a>)


`info point` Una dintre cele mai prestigioase conferințe din spațiul Machine Learning/AI este <a href = "https://recsys.acm.org/recsys23/">RecSys</a>. Deși e relativ tânără (17 ani), este cea mai importantă conferință din domeniul sistemelor de recomandare. Domeniul se pretează cercetării atât academice cât mai ales private având în vedere sumele consistente vehiculate pentru reclame online și a dorinței de a ghici intențiile unui internaut.

Sistemele de recomandare sunt algoritmi care oferă sugestii personalizate pentru articolele care sunt cele mai relevante pentru fiecare utilizator. Odată cu creșterea masivă a conținutului online disponibil, utilizatorii au fost inundați de opțiuni. Prin urmare, este crucial ca platformele web să ofere recomandări de articole fiecărui utilizator, pentru a crește satisfacția și implicarea utilizatorului.

Următoarea listă prezintă exemple de platforme web cunoscute cu un număr mare de conținut disponibil, care au nevoie de sisteme eficiente de recomandare pentru a menține interesul utilizatorilor (date din 2022).

- Youtube. În fiecare minut, oamenii încarcă 500 de ore de videoclipuri, de ex. ar dura 82 de ani pentru ca un utilizator să vizioneze toate videoclipurile încărcate doar în ultima oră.
- Spotify. Utilizatorii pot asculta peste 80 de milioane de melodii și podcasturi.
- Amazon. Utilizatorii pot cumpăra peste 350 de milioane de produse diferite.

La acestea se adaugă platformere de conținut video pe bază de abonament: Netflix, Amazon Prime, AppleTv, Disney+, HBOGO, Showtime. De asemenea conținutul din paginile de rețele sociale (Facebook, Instagram, X, etc.) este puternic influențat de profilul utilizatorilor.

Toate aceste platforme folosesc modele puternice de învățare automată pentru a genera recomandări relevante pentru fiecare utilizator.

### Feedback explicit vs feedback implicit
În sistemele de recomandare, modelele de învățare automată sunt folosite pentru a prezice evaluarea rᵤᵢ a unui utilizator u pentru un articol i. La momentul deducerii, recomandăm fiecărui utilizator u articolele l care au cel mai mare rating estimat rᵤᵢ.

Prin urmare, trebuie să colectăm feedback-ul utilizatorilor, astfel încât să putem avea un adevăr de bază pentru instruirea și evaluarea modelelor noastre. Aici trebuie făcută o distincție importantă între feedback-ul explicit și feedback-ul implicit.

<img style="float: center; padding-right: 10px; width: 600px" src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/feeback1.webp" >
<center><b>Fig. Feedback implicit și explicit</b> (Credits <a href ="https://towardsdatascience.com/recommender-systems-a-complete-guide-to-machine-learning-models-96d3f94ea748" >towardsdatascience</a>)</center>


<b>Feedback-ul explicit</b> este o evaluare acordată în mod explicit de către utilizator pentru a-și exprima satisfacția față de un articol. Exemple sunt: ​​numărul de stele pe o scară de la 1 la 5 dat după cumpărarea unui produs, degetul mare în sus/în jos după vizionarea unui videoclip etc. Acest feedback oferă informații detaliate despre cât de mult i-a plăcut unui utilizator un articol, dar este greu de colectați, deoarece majoritatea utilizatorilor de obicei nu scriu recenzii sau nu acordă evaluări explicite pentru fiecare articol pe care îl achiziționează.

<b>Feedback-ul implicit</b>, pe de altă parte, presupune că interacțiunile utilizator-articol sunt un indiciu al preferințelor. Exemple sunt: ​​istoricul de achiziții/navigare a unui utilizator, lista de melodii redate de un utilizator etc. Acest feedback este extrem de abundent, dar în același timp este mai puțin detaliat și mai zgomotos (de exemplu, cineva poate cumpăra un produs cadou pentru altcineva). Cu toate acestea, acest zgomot devine neglijabil în comparație cu dimensiunea mare a datelor disponibile de acest tip, iar majoritatea sistemelor moderne de recomandare tind să se bazeze pe feedback implicit.

Odată ce am colectat feedback-uri explicite sau implicite, putem crea matricea de evaluare a articolului utilizator rᵤᵢ. Pentru feedback explicit, fiecare intrare din rᵤᵢ este o valoare numerică - de ex. rᵤᵢ = „stele date de tine filmului i”—sau „?” dacă utilizatorul nu ați evaluat articolul i. Pentru feedback-ul implicit, valorile din rᵤᵢ sunt valori booleene care reprezintă prezența sau lipsa interacțiunii - de ex. rᵤᵢ = „Utilizatorul a văzut filmul i?”. Observați că matricea rᵤᵢ este foarte rară, deoarece utilizatorii interacționează cu puține articole dintre toate conținuturile disponibile și revizuiesc și mai puține articole!

<img style="float: center; padding-right: 10px; width: 600px" src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/ratings.webp" >
<center><b>Fig. Matricea rating-urilor</b> (Credits <a href ="https://towardsdatascience.com/recommender-systems-a-complete-guide-to-machine-learning-models-96d3f94ea748" >towardsdatascience</a>)</center>

### Abordări bazate pe conținut vs. Filtrare colaborativă
Sistemul de recomandare poate fi clasificat în funcție de tipul de informații utilizate pentru a prezice preferințele utilizatorului ca Filtrare bazată pe conținut sau Filtrare colaborativă.

<img style="float: center; padding-right: 10px; width: 600px" src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/filtering_types.webp" >
<center><b>Fig. Tipuri de Sisteme de recomandare</b> (Credits <a href ="https://towardsdatascience.com/recommender-systems-a-complete-guide-to-machine-learning-models-96d3f94ea748" >towardsdatascience</a>)</center>

#### Abordare bazată pe conținut
Metodele bazate pe conținut descriu utilizatorii și articolele prin metadatele lor cunoscute. Fiecare articol i este reprezentat de un set de etichete relevante - de ex. filmele platformei IMDb pot fi etichetate ca „acțiune”, „comedie”, etc. Fiecare utilizator u este reprezentat de un profil de utilizator, care poate fi creat din informații despre utilizator cunoscute, de ex. sex și vârstă – sau din activitatea trecută a utilizatorului.

Pentru a antrena un model de învățare automată cu această abordare, putem folosi un model k-NN (k-nearest neighbours = cei mai apropiați k vecini). De exemplu, dacă știm că utilizatorul a cumpărat un articol i, vă putem recomanda articolele disponibile cu caracteristici cele mai asemănătoare cu i.

<b>Avantajul acestei abordări</b> este că metadatele articolelor sunt cunoscute în avans, așa că le putem aplica și scenariilor Cold-Start în care un nou articol sau utilizator este adăugat pe platformă și nu avem interacțiuni utilizator-articol pentru a ne antrena modelul. <b>Dezavantajele</b> sunt că nu folosim întregul set de interacțiuni cunoscute utilizator-articol (fiecare utilizator este tratat independent) și că trebuie să cunoaștem informații despre metadate pentru fiecare articol și utilizator.

#### Abordare de filtrare colaborativă
Metodele de filtrare colaborativă nu folosesc metadatele articolului sau utilizatorului, ci încearcă în schimb să folosească feedback-urile sau istoricul activității tuturor utilizatorilor pentru a prezice evaluarea unui utilizator pentru un articol dat prin deducerea interdependențelor dintre utilizatori și articole din activitățile observate.

Pentru a antrena un model de învățare automată cu această abordare, încercăm de obicei să grupăm sau să <b>factorizăm matricea de rating rᵤᵢ</b> pentru a face predicții asupra perechilor neobservate (u,i), adică acolo unde rᵤᵢ = „?” mai sus. În continuarea vom discuta sumar algoritmul <b>Matrix Factorization</b>, care este cea mai populară metodă a acestei clase.

Avantajul acestei abordări este că se utilizează întregul set de interacțiuni utilizator-articol (adică matricea rᵤᵢ), ceea ce permite de obicei obținerea unei acuratețe mai mari decât utilizarea modelelor bazate pe conținut. Dezavantajul acestei abordări este că necesită câteva interacțiuni cu utilizatorul înainte ca modelul să poată fi montat.

#### Abordări hibride
În cele din urmă, există și <b>metode hibride</b> care încearcă să folosească atât metadatele cunoscute, cât și setul de interacțiuni observate utilizator-articol. Această abordare combină avantajele metodelor de filtrare bazată pe conținut și colaborativ și permite obținerea celor mai bune rezultate. Mai târziu în acest articol vă prezentăm LightFM, care este cel mai popular algoritm al acestei clase de metode.

#### Filtrare colaborativă: factorizarea matricei

Algoritmii de factorizare a unei matrice sunt probabil cele mai populare și eficiente metode de filtrare colaborativă pentru sistemele de recomandare. Factorizarea matriceală este un model cu factori latenți presupunând că pentru fiecare utilizator u și element i există reprezentări vectoriale latente pᵤ, qᵢ ∈ Rᶠ astfel încât rᵤᵢ poate fi exprimat în mod unic, adică „factorizat” - în termeni de pᵤ și qᵢ. Modulul Python <b>Surprise</b> de exemplu oferă implementări excelente ale acestor metode.

Algoritmul cel mai popular la ora actuală, care poate combina cele două metode de filtare, este SVD = Singular Value Decomposition.

Metodele de filtrare colaborative bazate pe factorizarea matricei produc adesea rezultate excelente, dar în scenariile de pornire la rece (cold start) — unde sunt nu prea sunt disponibile date de interacțiune pentru articole și utilizatori noi — nu pot face predicții bune, deoarece le lipsesc date pentru a estima factorii latenți. Abordările hibride rezolvă această problemă prin valorificarea metadatelor de articole sau utilizator cunoscute pentru a îmbunătăți modelul de factorizare matriceală. Modulul Python <b>LightFM</b> implementează unul dintre cei mai populari algoritmi hibrizi.

În LightFM, presupunem că pentru fiecare utilizator u am colectat un set de adnotări de etichetă Aᵁ(u) - de ex. „bărbat”, „vârsta < 30”, … — și, în mod similar, fiecare articol i are un set de adnotări Aᴵ(i) — de ex. „preț > 100 $”, „carte”, … Apoi modelăm fiecare etichetă de utilizator după un factor latent xᵁₐ ∈ Rᶠ și printr-un termen de părtinire (bias) bᵁₐ ∈ R și presupunem că reprezentarea vectorului utilizator pᵤ și biasul asociat bᵤ pot fi exprimat simplu ca suma acestor termeni xᵁₐ și, respectiv, bᵁₐ. Luăm aceeași abordare a etichetelor de articole, folosind factori latenți xᴵₐ ∈ Rᶠ și termeni de părtinire bᴵₐ ∈ R. Odată ce am definit pᵤ, qᵢ, bᵤ, bᵢ folosind aceste formule, putem folosi același model liniar al SVD pentru a descrie relația între acești termeni și rᵤᵢ, matricea de ratinguri.


Există trei cazuri interesante ale acestei abordări hibride a LightFM.

- <b>Pornire la rece/Cold start</b>. Dacă avem un nou articol i cu etichete cunoscute Aᴵ(i), atunci putem folosi vectorii latenți xᴵₐ (obținuți prin ajustarea modelului nostru pe datele anterioare) pentru a calcula încorporarea lui qᵢ și, prin urmare, pentru a estima pentru orice utilizator ratingul său rᵤᵢ .
- <b>Nu există etichete disponibile.</b> Dacă nu avem metadate cunoscute pentru articole sau utilizatori, singura adnotare pe care o putem folosi este o funcție de indicator, adică. o adnotare diferită pentru fiecare utilizator și fiecare articol. Apoi, matricele de caracteristici ale utilizatorului și ale articolului sunt matrice de identitate, iar LightFM se reduce la o metodă clasică de filtrare colaborativă, cum ar fi SVD.
- <b>Bazat pe conținut vs. hibrid</b>. Dacă am folosi doar etichete de utilizator sau de articol fără adnotări de indicator, LightFM ar fi aproape un model bazat pe conținut. Deci, în practică, pentru a valorifica interacțiunile utilizator-articol, adăugăm și etichetelor cunoscute o adnotare indicator diferită pentru fiecare utilizator și articol.

### Evaluarea sistemelor de recomandare

#### Sisteme explicite de recomandare cu feedback explicit
Acestea sunt sisteme în care utilizatorul oferă feedback explicit, de obicei sub forma unei evaluări numerice pentru fiecare recomandare.

Un exemplu este un sistem de recomandare de filme în care un utilizator ar putea acorda o evaluare numerică sub forma numărului de stele pentru fiecare film recomandat.

#### Valori utilizate în sistemele de recomandare explicită
Pentru un astfel de sistem, valorile utilizate ar putea fi destul de asemănătoare cu cele utilizate într-o problemă de regresie standard, deoarece ținta este într-adevăr un scor pe care l-ați putea prezice, iar scorul real este disponibil pentru a măsura cât de bună este predicția.

- Eroare absolută medie: medie pentru toate punctele de date, valoarea absolută a diferenței dintre evaluarea reală și evaluarea estimată.
- Root Mean Square Error: rădăcină pătrată a mediei peste toate punctele de date, pătratul diferenței dintre evaluarea reală și evaluarea estimată.

  
#### Sisteme de recomandare cu feedback implicit
Acestea sunt cele mai comune forme de sisteme de recomandare în care utilizatorului i se arată un set de recomandări și nu există feedback explicit. În schimb, există feedback implicit sub forma unui clic de către utilizator pe articol, sau adăugarea acestuia în coș și așa mai departe.

De exemplu, când vedeți un set de produse similare pe un site de comerț electronic, probabil că nu vi se cere o evaluare a calității recomandării! Mai degrabă, probabil că ați face clic pe unul sau mai multe dintre produse dacă vă plac. Și dacă dai clic pe unul dintre primele câteva produse, asta arată că și recomandatorul a reușit să-și ia comanda corect, pe lângă faptul că a afișat un produs relevant.

#### Valori utilizate în sistemele de recomandare implicită
Deoarece feedback-ul implicit disponibil este de obicei binar – sub forma „da itemul este relevant” sau „nu elementul nu este relevant”, metricile sunt similare cu cele ale algoritmilor de clasificare.

- Precizie: dintre toate articolele recomandate, câte sunt relevante?
- Recall: Dintre toate elementele relevante, câte au fost incluse în recomandări?
- Precision@K: Din primele K articole recomandate, câte sunt relevante?
- Recall@K: Dintre toate articolele relevante, câți au ajuns în primele K recomandări.
- Precizie medie medie (MAP): Dacă sunt recomandate K elemente, precizie@k este o medie de la i=1...K media pentru toate punctele de date.
- Mean Average Recall (MAR): Dacă sunt recomandate K elemente, recall@k este o medie de la i=1...K media pentru toate punctele de date.

- Discuții despre cum să aplici învățarea nesupervizată în propriul domeniu sau proiect.

## Concluzie: Recapitulare și resurse suplimentare

- Rezumat al principalelor concepte învățate în timpul lecției.

- Recomandări pentru resurse suplimentare și materiale de studiu.

## Puntea curs → seminar

**Concepte cheie**

1. **Distanța** între observații și de ce **scalarea** contează.
2. **PCA:** comprimăm multe coloane în câteva componente, păstrând cât mai multă varianță.
3. **k-means:** grupuri de observații asemănătoare; alegerea lui K (metoda cotului, silhouette).
4. **Reguli de asociere:** suport, încredere, lift (ce produse se cumpără împreună).

**Ce ceri asistentului AI** (exemplu de cerere)

> Pentru tabelul `general_data` din `lab-set2`, standardizează coloanele numerice (fără coloana țintă `Attrition` și fără ID-uri), aplică PCA și arată cât din varianță explică primele componente. Apoi aplică k-means pentru K de la 2 la 8 și afișează scorul silhouette pentru fiecare K.

**Ce verifici înainte să accepți rezultatul**

- [ ] datele au fost scalate înainte de PCA și k-means;
- [ ] ținta și coloanele de tip ID au fost excluse;
- [ ] grupurile sunt interpretate prin mediile coloanelor originale, nu doar prin numărul grupului.

Continuăm la seminar: [seminar-03-invatare-nesupervizata.ipynb](seminar-03-invatare-nesupervizata.ipynb).